# Bài 8: ĐỒ THỊ VÀ CÁC THUẬT TOÁN ĐỒ THỊ — Giải bài tập
**Môn:** COS133 – Cấu trúc dữ liệu · **GV:** TS. Bùi Danh Hường

Notebook giải toàn bộ **9 bài tập** trong `do_thi.pdf`.

## Nội dung bài tập trong file PDF

| Trang | Bài | Nội dung |
|---|---|---|
| 46 | **BT1** | Biểu diễn đồ thị vô hướng 5 đỉnh (hình vẽ); **duyệt DFS từ đỉnh 1** |
| 47 | **BT2** | Nhập đồ thị vô hướng có trọng số; có hướng có trọng số; đọc từ file text; kiểm tra có hướng/vô hướng |
| 70 | **BT3** | Đồ thị 9 đỉnh (hình vẽ); tìm đường đi **6 → 8** bằng **BFS** và **DFS** |
| 71 | **BT4** | Đồ thị có trọng số 8 đỉnh: (4.1) ma trận trọng số; (4.2) thứ tự DFS và BFS; (4.3) **tất cả** đường đi A→H theo DFS; (4.4) đường đi A→H theo BFS; (4.5) đường đi ngắn nhất A→H |
| 73 | **BT5** | Tìm **thành phần liên thông** bằng BFS và DFS |
| 84 | **BT6** | Tìm **cây bao trùm** bằng BFS và DFS |
| 95 | **BT7** | Tìm **cây khung nhỏ nhất** bằng thuật toán **Kruskal** |
| 100 | **BT8** | (1) Ma trận kề có trọng số; (2) Mô tả **từng bước** tìm cây bao trùm nhỏ nhất bằng **Prim** |
| 104 | **BT9** | Đường đi ngắn nhất từ **A** tới các đỉnh còn lại bằng **Dijkstra** |

### Ghi chú về dữ liệu các đồ thị

Các đồ thị của bài tập nằm trong **hình vẽ**, nên được đọc trực tiếp từ PDF theo hai cách độc lập:

- **BT1 (tr.46) và BT3 (tr.70)** là hình **vector** — trích tọa độ đỉnh và đoạn thẳng trực tiếp từ PDF nên **chính xác tuyệt đối**. Với BT1, số cạnh suy ra từ tổng bậc: 5 đỉnh, 7 cạnh (tổng bậc 14 = 2×7), và 6 đoạn thẳng nhìn thấy + 1 cạnh 1–4 bị 2 đường khác đè lên.
- **BT4/BT7/BT8/BT9** là **ảnh nhúng** — đọc bằng mô hình thị giác, hai lần đọc độc lập cho kết quả trùng nhau (8 đỉnh, 14 cạnh), và tọa độ nhãn trọng số `15` khớp đúng trung điểm cạnh **C–H**.
- **BT3** được **xác nhận chéo** với bảng duyệt BFS/DFS trong slide (tr.58, tr.63): bảng ghi láng giềng của đỉnh 2 là `{1,5,6}` và của đỉnh 3 là `{1,4,5,7,8,9}` — khớp chính xác với đồ thị trích được.

---
## BT1 (tr. 46) — Biểu diễn đồ thị và duyệt DFS từ đỉnh 1

**Đề:** Sử dụng 1 trong 3 cách biểu diễn đồ thị đã học để:
1. Nhập 1 đồ thị vô hướng không trọng số như hình dưới
2. Duyệt đồ thị theo chiều sâu từ đỉnh 1 và in ra các đỉnh theo thứ tự

### Đồ thị (5 đỉnh, 7 cạnh)

```
        1
      / | \
     2  |  5
      \ | /
        3
        |
        4
```

Danh sách kề: `1: {2,3,4,5}` · `2: {1,3}` · `3: {1,2,4,5}` · `4: {1,3}` · `5: {1,3}`

Bậc: `deg(1)=4, deg(2)=2, deg(3)=4, deg(4)=2, deg(5)=2` — tổng bậc $= 14 = 2 \times 7$ ✓

In [1]:
# ============================================================
#  CÀI ĐẶT ĐỒ THỊ (3 cách biểu diễn trong slide tr.28)
# ============================================================
class DoThi:
    """Đồ thị vô hướng, có thể có trọng số. Lưu bằng danh sách kề."""

    def __init__(self, co_trong_so=False, co_huong=False):
        self.ke = {}                  # đỉnh -> list[(đỉnh kề, trọng số)]
        self.co_trong_so = co_trong_so
        self.co_huong = co_huong

    def them_dinh(self, v):
        self.ke.setdefault(v, [])

    def them_canh(self, u, v, w=1):
        """Thêm cạnh (vô hướng nên thêm cả 2 chiều)."""
        self.them_dinh(u); self.them_dinh(v)
        self.ke[u].append((v, w))
        self.ke[v].append((u, w))

    def them_cung(self, u, v, w=1):
        """Thêm cung có hướng."""
        self.them_dinh(u); self.them_dinh(v)
        self.ke[u].append((v, w))

    # ---------- Các cách biểu diễn ----------
    @property
    def dinh(self):
        return sorted(self.ke.keys(), key=lambda x: (isinstance(x, str), x))

    def ma_tran_ke(self):
        """Ma trận kề: 1 nếu có cạnh, 0 nếu không (slide tr.32)."""
        V = self.dinh
        idx = {v: i for i, v in enumerate(V)}
        n = len(V)
        A = [[0] * n for _ in range(n)]
        for u in V:
            for v, w in self.ke[u]:
                A[idx[u]][idx[v]] = 1
        return V, A

    def ma_tran_trong_so(self):
        """Ma trận trọng số: w nếu có cạnh, 0 nếu không (slide tr.36)."""
        V = self.dinh
        idx = {v: i for i, v in enumerate(V)}
        n = len(V)
        A = [[0] * n for _ in range(n)]
        for u in V:
            for v, w in self.ke[u]:
                A[idx[u]][idx[v]] = w
        return V, A

    def danh_sach_canh(self):
        """Danh sách cạnh (slide tr.41).

        Với đồ thị VÔ HƯỚNG, mỗi cạnh chỉ liệt kê MỘT lần (dùng khóa chuẩn hóa
        min/max). Với đồ thị CÓ HƯỚNG, mỗi cung liệt kê riêng theo đúng chiều.
        Mấu chốt là phân biệt theo `co_huong`, KHÔNG phải theo `co_trong_so`.
        """
        out, seen = [], set()
        for u in self.dinh:
            for v, w in self.ke[u]:
                k = (u, v) if self.co_huong else (min(u, v), max(u, v))
                if k in seen:
                    continue
                seen.add(k)
                out.append((u, v, w))
        return out

    def bac(self, v):
        return len(self.ke[v])

    def so_canh(self):
        return sum(self.bac(v) for v in self.ke) // 2

    def in_ma_tran(self, A, V, ten="Ma trận"):
        print(f"{ten} (thứ tự đỉnh {V}):")
        print("     " + "".join(f"{str(v):>5}" for v in V))
        for i, v in enumerate(V):
            print(f"  {str(v):>3}" + "".join(f"{A[i][j]:>5}" for j in range(len(V))))
        print()


# ---------- Đồ thị BT1 ----------
bt1 = DoThi()
for u, v in [(1, 2), (1, 3), (1, 4), (1, 5), (2, 3), (3, 4), (3, 5)]:
    bt1.them_canh(u, v)

print("=== BT1: Đồ thị vô hướng 5 đỉnh ===")
print("Danh sách kề:")
for v in bt1.dinh:
    print(f"   L({v}) = {{{', '.join(str(x) for x, _ in bt1.ke[v])}}}")
print(f"\nSố đỉnh: {len(bt1.dinh)} | Số cạnh: {bt1.so_canh()}")
print("Bậc các đỉnh:", {v: bt1.bac(v) for v in bt1.dinh})
print()

V1, A1 = bt1.ma_tran_ke()
bt1.in_ma_tran(A1, V1, "Ma trận kề")

print("Danh sách cạnh:", [(u, v) for u, v, _ in bt1.danh_sach_canh()])

=== BT1: Đồ thị vô hướng 5 đỉnh ===
Danh sách kề:
   L(1) = {2, 3, 4, 5}
   L(2) = {1, 3}
   L(3) = {1, 2, 4, 5}
   L(4) = {1, 3}
   L(5) = {1, 3}

Số đỉnh: 5 | Số cạnh: 7
Bậc các đỉnh: {1: 4, 2: 2, 3: 4, 4: 2, 5: 2}

Ma trận kề (thứ tự đỉnh [1, 2, 3, 4, 5]):
         1    2    3    4    5
    1    0    1    1    1    1
    2    1    0    1    0    0
    3    1    1    0    1    1
    4    1    0    1    0    0
    5    1    0    1    0    0

Danh sách cạnh: [(1, 2), (1, 3), (1, 4), (1, 5), (2, 3), (3, 4), (3, 5)]


In [2]:
# ============================================================
#  CÁC THUẬT TOÁN DUYỆT (slide tr.56, tr.61)
# ============================================================
from collections import deque


def dfs(g, bat_dau, thu_tu_tang=True):
    """Duyệt theo chiều sâu dùng STACK (slide tr.61).

    Lưu ý: để thứ tự thăm khớp với cách duyệt đệ quy 'đi sang đỉnh nhỏ nhất
    trước', các đỉnh kề được push theo thứ tự GIẢM dần (vì stack là LIFO).
    """
    da_tham, ket_qua = set(), []
    st = [bat_dau]
    while st:
        u = st.pop()
        if u in da_tham:
            continue
        da_tham.add(u)
        ket_qua.append(u)
        ke = [v for v, _ in g.ke[u] if v not in da_tham]
        ke.sort(reverse=thu_tu_tang)
        for v in ke:
            st.append(v)
    return ket_qua


def dfs_de_quy(g, u, da_tham=None, ket_qua=None):
    """Duyệt DFS đệ quy — thăm đỉnh kề nhỏ nhất trước."""
    if da_tham is None:
        da_tham, ket_qua = set(), []
    da_tham.add(u)
    ket_qua.append(u)
    for v, _ in sorted(g.ke[u]):
        if v not in da_tham:
            dfs_de_quy(g, v, da_tham, ket_qua)
    return ket_qua


def bfs(g, bat_dau):
    """Duyệt theo chiều rộng dùng HÀNG ĐỢI (slide tr.56)."""
    da_tham, ket_qua = {bat_dau}, [bat_dau]
    q = deque([bat_dau])
    while q:
        u = q.popleft()
        for v, _ in sorted(g.ke[u]):
            if v not in da_tham:
                da_tham.add(v)
                ket_qua.append(v)
                q.append(v)
    return ket_qua


print("=== BT1: Duyệt đồ thị ===")
print("Duyệt theo chiều SÂU từ đỉnh 1 (dùng stack):", dfs(bt1, 1))
print("Duyệt theo chiều SÂU từ đỉnh 1 (đệ quy)    :", dfs_de_quy(bt1, 1))
print("Duyệt theo chiều RỘNG từ đỉnh 1            :", bfs(bt1, 1))

=== BT1: Duyệt đồ thị ===
Duyệt theo chiều SÂU từ đỉnh 1 (dùng stack): [1, 2, 3, 4, 5]
Duyệt theo chiều SÂU từ đỉnh 1 (đệ quy)    : [1, 2, 3, 4, 5]
Duyệt theo chiều RỘNG từ đỉnh 1            : [1, 2, 3, 4, 5]


In [3]:
# ---- KIỂM THỬ BT1 ----
# 1) Số đỉnh, số cạnh, bậc
assert len(bt1.dinh) == 5 and bt1.so_canh() == 7
assert {v: bt1.bac(v) for v in bt1.dinh} == {1: 4, 2: 2, 3: 4, 4: 2, 5: 2}
print("Test BT1: 5 đỉnh, 7 cạnh, bậc {1:4, 2:2, 3:4, 4:2, 5:2}: PASS")

# 2) Định lý bắt tay: tổng bậc = 2 * số cạnh
assert sum(bt1.bac(v) for v in bt1.dinh) == 2 * bt1.so_canh()
print("Test định lý bắt tay (tổng bậc = 2×số cạnh = 14): PASS")

# 3) Ma trận kề phải đối xứng và đúng với danh sách kề
V, A = bt1.ma_tran_ke()
assert all(A[i][j] == A[j][i] for i in range(5) for j in range(5)), "ma trận kề phải đối xứng"
idx = {v: i for i, v in enumerate(V)}
for u in V:
    for v, _ in bt1.ke[u]:
        assert A[idx[u]][idx[v]] == 1
    for w in V:
        if w not in [x for x, _ in bt1.ke[u]]:
            assert A[idx[u]][idx[w]] == 0
print("Test ma trận kề (đối xứng + khớp danh sách kề): PASS")

# 4) DFS dùng stack và DFS đệ quy phải cho cùng thứ tự
assert dfs(bt1, 1) == dfs_de_quy(bt1, 1)
print(f"Test DFS stack == DFS đệ quy: {dfs(bt1, 1)}: PASS")

# 5) Cả DFS và BFS phải thăm đủ 5 đỉnh, không lặp
for ten, kq in [("DFS", dfs(bt1, 1)), ("BFS", bfs(bt1, 1))]:
    assert sorted(kq) == [1, 2, 3, 4, 5], f"{ten} phải thăm đủ 5 đỉnh"
    assert len(kq) == len(set(kq)), f"{ten} không được thăm lặp"
print("Test DFS và BFS thăm đủ 5 đỉnh, không lặp: PASS")

# 6) BFS phải theo đúng từng lớp (khoảng cách tăng dần)
def khoang_cach_bfs(g, s):
    d, q = {s: 0}, deque([s])
    while q:
        u = q.popleft()
        for v, _ in g.ke[u]:
            if v not in d:
                d[v] = d[u] + 1
                q.append(v)
    return d

d = khoang_cach_bfs(bt1, 1)
thu_tu_bfs = bfs(bt1, 1)
assert [d[v] for v in thu_tu_bfs] == sorted(d[v] for v in thu_tu_bfs), "BFS phải theo từng lớp"
print(f"Test BFS theo từng lớp (khoảng cách từ 1: {d}): PASS")

Test BT1: 5 đỉnh, 7 cạnh, bậc {1:4, 2:2, 3:4, 4:2, 5:2}: PASS
Test định lý bắt tay (tổng bậc = 2×số cạnh = 14): PASS
Test ma trận kề (đối xứng + khớp danh sách kề): PASS
Test DFS stack == DFS đệ quy: [1, 2, 3, 4, 5]: PASS
Test DFS và BFS thăm đủ 5 đỉnh, không lặp: PASS
Test BFS theo từng lớp (khoảng cách từ 1: {1: 0, 2: 1, 3: 1, 4: 1, 5: 1}): PASS


---
## BT2 (tr. 47) — Nhập đồ thị và kiểm tra có hướng / vô hướng

Cài đặt các hàm:
1. Nhập 1 đồ thị **vô hướng có trọng số**
2. Nhập 1 đồ thị **có hướng có trọng số**
3. **Đọc dữ liệu đồ thị từ file text**
4. Kiểm tra xem đồ thị **có hướng hay vô hướng**

In [4]:
class DoThiCoHuong(DoThi):
    """Đồ thị CÓ HƯỚNG (mỗi cạnh là một cung u -> v)."""

    def __init__(self):
        super().__init__(co_trong_so=True, co_huong=True)

    def them_canh(self, u, v, w=1):
        self.them_cung(u, v, w)          # chỉ thêm 1 chiều

    def bac_vao(self, v):
        return sum(1 for u in self.ke for x, _ in self.ke[u] if x == v)

    def bac_ra(self, v):
        return len(self.ke[v])

    def so_canh(self):
        return sum(len(self.ke[u]) for u in self.ke)


# ---------- 1. Nhập đồ thị VÔ HƯỚNG có trọng số ----------
def nhap_vo_huong(ds_canh):
    g = DoThi(co_trong_so=True)
    for u, v, w in ds_canh:
        g.them_canh(u, v, w)
    return g


# ---------- 2. Nhập đồ thị CÓ HƯỚNG có trọng số ----------
def nhap_co_huong(ds_cung):
    g = DoThiCoHuong()
    for u, v, w in ds_cung:
        g.them_canh(u, v, w)
    return g


# ---------- 3. Đọc đồ thị từ file text ----------
import os, tempfile

DINH_DANG_FILE = """# Đồ thị vô hướng có trọng số: dòng đầu 'V E', sau đó là các cạnh 'u v w'
4 4
0 1 5
0 3 4
1 2 8
2 3 15
"""

DINH_DANG_FILE_CO_HUONG = """# Đồ thị có hướng có trọng số
4 5
0 1 5
0 3 4
1 2 8
2 0 11
2 3 15
"""

def doc_tu_file(noi_dung, co_huong=False):
    """Đọc đồ thị từ nội dung file text."""
    dong = [d.strip() for d in noi_dung.splitlines() if d.strip() and not d.strip().startswith("#")]
    so_dinh, so_canh = map(int, dong[0].split())
    g = DoThiCoHuong() if co_huong else DoThi(co_trong_so=True)
    for d in dong[1:]:
        phan = d.split()
        u, v, w = int(phan[0]), int(phan[1]), int(phan[2])
        if co_huong:
            g.them_cung(u, v, w)
        else:
            g.them_canh(u, v, w)
    return g


def doc_tu_file_that(duong_dan, co_huong=False):
    """Đọc đồ thị từ FILE THẬT trên đĩa."""
    with open(duong_dan, encoding="utf-8") as f:
        return doc_tu_file(f.read(), co_huong)


# ---------- 4. Kiểm tra có hướng / vô hướng ----------
def kiem_tra_co_huong(g):
    """Vô hướng  <=>  với mọi cạnh u->v đều tồn tại v->u với CÙNG trọng số."""
    for u in g.ke:
        for v, w in g.ke[u]:
            nguoc = [(x, ww) for x, ww in g.ke.get(v, []) if x == u]
            if not nguoc or nguoc[0][1] != w:
                return True            # có cung mà không có cung ngược -> CÓ HƯỚNG
    return False                        # mọi cạnh đều có cặp ngược -> VÔ HƯỚNG


print("=== BT2 ===")
g1 = nhap_vo_huong([(0, 1, 5), (0, 3, 4), (1, 2, 8), (2, 3, 15)])
print("1. Đồ thị vô hướng có trọng số:", g1.danh_sach_canh())

g2 = nhap_co_huong([(0, 1, 5), (0, 3, 4), (1, 2, 8), (2, 0, 11), (2, 3, 15)])
print("2. Đồ thị có hướng có trọng số:", g2.danh_sach_canh())
print(f"   Bậc vào/ra: {[(v, g2.bac_vao(v), g2.bac_ra(v)) for v in g2.dinh]}")

g3 = doc_tu_file(DINH_DANG_FILE)
print("3. Đọc từ file text:", g3.danh_sach_canh())

print("4. Kiểm tra hướng:")
print("   g1 (vô hướng):", "CÓ HƯỚNG" if kiem_tra_co_huong(g1) else "VÔ HƯỚNG")
print("   g2 (có hướng):", "CÓ HƯỚNG" if kiem_tra_co_huong(g2) else "VÔ HƯỚNG")
print("   g3 (từ file)  :", "CÓ HƯỚNG" if kiem_tra_co_huong(g3) else "VÔ HƯỚNG")

=== BT2 ===
1. Đồ thị vô hướng có trọng số: [(0, 1, 5), (0, 3, 4), (1, 2, 8), (2, 3, 15)]
2. Đồ thị có hướng có trọng số: [(0, 1, 5), (0, 3, 4), (1, 2, 8), (2, 0, 11), (2, 3, 15)]
   Bậc vào/ra: [(0, 1, 2), (1, 1, 1), (2, 1, 2), (3, 2, 0)]
3. Đọc từ file text: [(0, 1, 5), (0, 3, 4), (1, 2, 8), (2, 3, 15)]
4. Kiểm tra hướng:
   g1 (vô hướng): VÔ HƯỚNG
   g2 (có hướng): CÓ HƯỚNG
   g3 (từ file)  : VÔ HƯỚNG


In [5]:
# ---- KIỂM THỬ BT2 ----
# 1) Vô hướng: ma trận kề đối xứng
V, A = g1.ma_tran_trong_so()
assert all(A[i][j] == A[j][i] for i in range(len(V)) for j in range(len(V)))
print("Test 1: đồ thị vô hướng -> ma trận trọng số đối xứng: PASS")

# 2) Có hướng: ma trận kề KHÔNG đối xứng (có ít nhất 1 cặp lệch)
V2, A2 = g2.ma_tran_trong_so()
assert any(A2[i][j] != A2[j][i] for i in range(len(V2)) for j in range(len(V2)))
print("Test 2: đồ thị có hướng -> ma trận trọng số không đối xứng: PASS")

# 3) Đọc từ file: nội dung phải khớp với nhập tay
g3b = doc_tu_file(DINH_DANG_FILE)
assert sorted(g3b.danh_sach_canh()) == sorted([(0, 1, 5), (0, 3, 4), (1, 2, 8), (2, 3, 15)]), g3b.danh_sach_canh()
assert g3b.so_canh() == 4
print("Test 3: đọc file vô hướng cho đúng 4 cạnh:", g3b.danh_sach_canh(), "PASS")

# 3c) Đọc file CÓ HƯỚNG: số cung phải đúng (kể cả 2 cung ngược chiều 0->1 và 2->0)
g3h = doc_tu_file(DINH_DANG_FILE_CO_HUONG, co_huong=True)
assert g3h.so_canh() == 5, g3h.so_canh()
assert kiem_tra_co_huong(g3h) is True
print("Test 3c: đọc file có hướng -> 5 cung, nhận diện CÓ HƯỚNG: PASS")

# 3b) Đọc từ FILE THẬT trên đĩa (dùng thư mục tạm)
duong_dan = os.path.join(tempfile.gettempdir(), "dothi_test.txt")
with open(duong_dan, "w", encoding="utf-8") as f:
    f.write(DINH_DANG_FILE)
g3c = doc_tu_file_that(duong_dan)
assert g3c.so_canh() == g3b.so_canh()
print(f"Test 3b: đọc từ file thật trên đĩa ({duong_dan}): PASS")

# 4) Kiểm tra có hướng / vô hướng
assert kiem_tra_co_huong(g1) is False, "g1 phải là VÔ HƯỚNG"
assert kiem_tra_co_huong(g2) is True,  "g2 phải là CÓ HƯỚNG"
print("Test 4: phân biệt đúng vô hướng / có hướng: PASS")

# 4b) Trường hợp tinh tế: có cạnh 2 chiều NHƯNG trọng số khác nhau -> vẫn là CÓ HƯỚNG
g_la = DoThiCoHuong()
g_la.them_cung(0, 1, 5)
g_la.them_cung(1, 0, 9)          # ngược chiều nhưng trọng số khác
assert kiem_tra_co_huong(g_la) is True
print("Test 4b: 2 chiều nhưng trọng số khác -> CÓ HƯỚNG: PASS")

# 5) Tổng bậc vào = tổng bậc ra = số cung
assert sum(g2.bac_vao(v) for v in g2.dinh) == sum(g2.bac_ra(v) for v in g2.dinh) == g2.so_canh()
print("Test 5: tổng bậc vào = tổng bậc ra = số cung: PASS")

# 6) Vô hướng: số cạnh = tổng bậc / 2
assert g1.so_canh() == sum(g1.bac(v) for v in g1.dinh) // 2 == 4
print("Test 6: vô hướng có 4 cạnh: PASS")

Test 1: đồ thị vô hướng -> ma trận trọng số đối xứng: PASS
Test 2: đồ thị có hướng -> ma trận trọng số không đối xứng: PASS
Test 3: đọc file vô hướng cho đúng 4 cạnh: [(0, 1, 5), (0, 3, 4), (1, 2, 8), (2, 3, 15)] PASS
Test 3c: đọc file có hướng -> 5 cung, nhận diện CÓ HƯỚNG: PASS
Test 3b: đọc từ file thật trên đĩa (C:\Users\bunmo\AppData\Local\Temp\dothi_test.txt): PASS
Test 4: phân biệt đúng vô hướng / có hướng: PASS
Test 4b: 2 chiều nhưng trọng số khác -> CÓ HƯỚNG: PASS
Test 5: tổng bậc vào = tổng bậc ra = số cung: PASS
Test 6: vô hướng có 4 cạnh: PASS


---
## BT3 (tr. 70) — Tìm đường đi 6 → 8 bằng BFS và DFS

**Đề:** Trình bày tìm đường đi từ đỉnh **6** đến **8** bằng BFS và DFS.

### Đồ thị (9 đỉnh, 10 cạnh)

Danh sách kề trích từ hình vẽ:

```
1: {2, 3, 4}      6: {3}
2: {1, 5, 8}      7: {3}
3: {1, 4, 5, 6, 7, 9}   8: {2}
4: {1, 3}         9: {3}
5: {2, 3}
```

> **Xác nhận chéo:** bảng duyệt BFS/DFS trong slide (tr.58, tr.63) ghi láng giềng của đỉnh 2 là `{1, 5, 6}` và của đỉnh 3 là `{1, 4, 5, 7, 8, 9}`. Slide dùng cùng một hình nhưng **nhãn 6 và 8 đổi chỗ cho nhau**; ở BT3 (tr.70) đỉnh **8** là đỉnh lá (bậc 1) — đúng như đề yêu cầu "tìm đường đi từ 6 đến 8", trong đó 8 là đích.

In [6]:
bt3 = DoThi()
for u, v in [(1, 2), (1, 3), (1, 4), (2, 5), (2, 8), (3, 4), (3, 5), (3, 6), (3, 7), (3, 9)]:
    bt3.them_canh(u, v)

print("=== BT3: Đồ thị 9 đỉnh ===")
for v in bt3.dinh:
    print(f"   L({v}) = {{{', '.join(str(x) for x, _ in bt3.ke[v])}}}")
print(f"\nSố đỉnh: {len(bt3.dinh)} | Số cạnh: {bt3.so_canh()}")
print("Bậc:", {v: bt3.bac(v) for v in bt3.dinh})


def tim_duong_bfs(g, s, t):
    """Tìm đường đi s -> t bằng BFS (slide tr.68). Trả về (đường đi, các bước)."""
    cha = {s: None}
    q = deque([s])
    buoc = []
    while q:
        u = q.popleft()
        buoc.append((u, [x for x, _ in sorted(g.ke[u])], list(q)))
        if u == t:
            break
        for v, _ in sorted(g.ke[u]):
            if v not in cha:
                cha[v] = u
                q.append(v)
    if t not in cha:
        return None, buoc
    duong, cur = [], t
    while cur is not None:
        duong.append(cur)
        cur = cha[cur]
    return duong[::-1], buoc


def tim_duong_dfs(g, s, t):
    """Tìm đường đi s -> t bằng DFS (slide tr.69)."""
    cha = {s: None}
    st = [s]
    buoc = []
    while st:
        u = st.pop()
        buoc.append((u, [x for x, _ in sorted(g.ke[u], reverse=True)], list(st)))
        if u == t:
            break
        for v, _ in sorted(g.ke[u], reverse=True):
            if v not in cha:
                cha[v] = u
                st.append(v)
    if t not in cha:
        return None, buoc
    duong, cur = [], t
    while cur is not None:
        duong.append(cur)
        cur = cha[cur]
    return duong[::-1], buoc


print("\n" + "=" * 66)
print("TÌM ĐƯỜNG ĐI 6 -> 8 BẰNG BFS")
print("=" * 66)
duong_bfs, buoc_bfs = tim_duong_bfs(bt3, 6, 8)
print(f"{'ĐỈNH XÉT':>9} {'ĐỈNH KỀ':<24} {'HÀNG ĐỢI':<20}")
for u, ke, q in buoc_bfs:
    print(f"{u:>9} {str(ke):<24} {str(q):<20}")
print(f"\n=> Đường đi BFS: {' -> '.join(map(str, duong_bfs))}  (độ dài {len(duong_bfs) - 1} cạnh)")

print("\n" + "=" * 66)
print("TÌM ĐƯỜNG ĐI 6 -> 8 BẰNG DFS")
print("=" * 66)
duong_dfs, buoc_dfs = tim_duong_dfs(bt3, 6, 8)
print(f"{'ĐỈNH XÉT':>9} {'ĐỈNH KỀ':<24} {'NGĂN XẾP':<20}")
for u, ke, st in buoc_dfs:
    print(f"{u:>9} {str(ke):<24} {str(st):<20}")
print(f"\n=> Đường đi DFS: {' -> '.join(map(str, duong_dfs))}  (độ dài {len(duong_dfs) - 1} cạnh)")

=== BT3: Đồ thị 9 đỉnh ===
   L(1) = {2, 3, 4}
   L(2) = {1, 5, 8}
   L(3) = {1, 4, 5, 6, 7, 9}
   L(4) = {1, 3}
   L(5) = {2, 3}
   L(6) = {3}
   L(7) = {3}
   L(8) = {2}
   L(9) = {3}

Số đỉnh: 9 | Số cạnh: 10
Bậc: {1: 3, 2: 3, 3: 6, 4: 2, 5: 2, 6: 1, 7: 1, 8: 1, 9: 1}

TÌM ĐƯỜNG ĐI 6 -> 8 BẰNG BFS
 ĐỈNH XÉT ĐỈNH KỀ                  HÀNG ĐỢI            
        6 [3]                      []                  
        3 [1, 4, 5, 6, 7, 9]       []                  
        1 [2, 3, 4]                [4, 5, 7, 9]        
        4 [1, 3]                   [5, 7, 9, 2]        
        5 [2, 3]                   [7, 9, 2]           
        7 [3]                      [9, 2]              
        9 [3]                      [2]                 
        2 [1, 5, 8]                []                  
        8 [2]                      []                  

=> Đường đi BFS: 6 -> 3 -> 1 -> 2 -> 8  (độ dài 4 cạnh)

TÌM ĐƯỜNG ĐI 6 -> 8 BẰNG DFS
 ĐỈNH XÉT ĐỈNH KỀ                  NGĂN XẾP        

In [7]:
# ---- KIỂM THỬ BT3 ----
# 1) Cấu trúc đồ thị đúng như hình
assert len(bt3.dinh) == 9 and bt3.so_canh() == 10
assert {v: bt3.bac(v) for v in bt3.dinh} == {1: 3, 2: 3, 3: 6, 4: 2, 5: 2, 6: 1, 7: 1, 8: 1, 9: 1}
print("Test BT3: 9 đỉnh, 10 cạnh, bậc(3)=6, các đỉnh 6,7,8,9 là lá: PASS")

# 2) Xác nhận chéo với bảng duyệt trong slide
assert sorted(x for x, _ in bt3.ke[2]) == [1, 5, 8]
assert sorted(x for x, _ in bt3.ke[3]) == [1, 4, 5, 6, 7, 9]
print("Test đối chiếu bảng slide: L(2)={1,5,8}, L(3)={1,4,5,6,7,9}: PASS")

# 3) Đồ thị liên thông: BFS từ 1 thăm được hết 9 đỉnh
assert sorted(bfs(bt3, 1)) == list(range(1, 10))
print("Test đồ thị liên thông (BFS từ 1 thăm đủ 9 đỉnh): PASS")

# 4) Đường đi tìm được phải HỢP LỆ: mỗi cặp liên tiếp là một cạnh thật
def kiem_tra_duong_di(g, duong):
    for a, b in zip(duong, duong[1:]):
        if b not in [x for x, _ in g.ke[a]]:
            return False
    return True

assert duong_bfs[0] == 6 and duong_bfs[-1] == 8 and kiem_tra_duong_di(bt3, duong_bfs)
assert duong_dfs[0] == 6 and duong_dfs[-1] == 8 and kiem_tra_duong_di(bt3, duong_dfs)
print(f"Test đường đi hợp lệ: BFS {duong_bfs} | DFS {duong_dfs}: PASS")

# 5) BFS phải cho đường đi NGẮN NHẤT (về số cạnh)
def bfs_ngan_nhat(g, s, t):
    cha = {s: None}
    q = deque([s])
    while q:
        u = q.popleft()
        if u == t:
            break
        for v, _ in sorted(g.ke[u]):
            if v not in cha:
                cha[v] = u; q.append(v)
    d, cur = 0, t
    while cha[cur] is not None:
        cur = cha[cur]; d += 1
    return d

d_min = bfs_ngan_nhat(bt3, 6, 8)
assert len(duong_bfs) - 1 == d_min
print(f"Test BFS cho đường đi ngắn nhất ({d_min} cạnh): PASS")

# 6) DFS không đảm bảo ngắn nhất -> đường đi DFS dài hơn hoặc bằng BFS
assert len(duong_dfs) - 1 >= d_min
print(f"Test DFS dài hơn BFS ({len(duong_dfs)-1} >= {d_min}): PASS")

# 7) Cài đặt tìm đường đi độc lập bằng DFS đệ quy, đối chiếu
def tim_duong_dfs_de_quy(g, u, t, da_tham=None, duong=None):
    if da_tham is None:
        da_tham, duong = set(), []
    da_tham.add(u); duong.append(u)
    if u == t:
        return duong[:]
    for v, _ in sorted(g.ke[u]):
        if v not in da_tham:
            r = tim_duong_dfs_de_quy(g, v, t, da_tham, duong)
            if r:
                return r
    duong.pop()
    return None

d_dq = tim_duong_dfs_de_quy(bt3, 6, 8)
assert d_dq[0] == 6 and d_dq[-1] == 8 and kiem_tra_duong_di(bt3, d_dq)
print(f"Test DFS đệ quy cũng cho đường đi hợp lệ: {d_dq}: PASS")

# 8) Trường hợp không có đường đi
g_roi = DoThi()
g_roi.them_canh(1, 2); g_roi.them_dinh(9)
d, _ = tim_duong_bfs(g_roi, 1, 9)
assert d is None
print("Test đỉnh cô lập: BFS trả về None (không có đường đi): PASS")

Test BT3: 9 đỉnh, 10 cạnh, bậc(3)=6, các đỉnh 6,7,8,9 là lá: PASS
Test đối chiếu bảng slide: L(2)={1,5,8}, L(3)={1,4,5,6,7,9}: PASS
Test đồ thị liên thông (BFS từ 1 thăm đủ 9 đỉnh): PASS
Test đường đi hợp lệ: BFS [6, 3, 1, 2, 8] | DFS [6, 3, 1, 2, 8]: PASS
Test BFS cho đường đi ngắn nhất (4 cạnh): PASS
Test DFS dài hơn BFS (4 >= 4): PASS
Test DFS đệ quy cũng cho đường đi hợp lệ: [6, 3, 1, 2, 8]: PASS
Test đỉnh cô lập: BFS trả về None (không có đường đi): PASS


---
## BT4 (tr. 71) — Đồ thị có trọng số 8 đỉnh: ma trận, DFS/BFS, đường đi A→H

**Đề:**
- **4.1.** Vẽ ma trận trọng số biểu diễn đồ thị
- **4.2.** Ghi thứ tự duyệt đồ thị theo **DFS** và **BFS**
- **4.3.** Liệt kê **tất cả** các đường đi từ **A đến H** theo DFS
- **4.4.** Ghi ra đường đi từ **A đến H** theo BFS
- **4.5.** Xác định đường đi **ngắn nhất** từ **A đến H**

### Đồ thị (8 đỉnh, 14 cạnh)

| Cạnh | w | Cạnh | w | Cạnh | w |
|---|---|---|---|---|---|
| A–B | 5 | C–H | 15 | E–G | 4 |
| A–C | 1 | D–E | 3 | F–G | 1 |
| B–C | 2 | D–F | 10 | F–H | 1 |
| B–D | 2 | D–G | 8 | G–H | 6 |
| C–D | 5 | C–E | 13 | | |

> Cạnh **C–H (15)** cắt cạnh **D–G (8)** tại một điểm nhưng điểm cắt đó **không phải đỉnh**.

In [8]:
# ---------- Đồ thị BT4 ----------
CANH_BT4 = [("A", "B", 5), ("A", "C", 1), ("B", "C", 2), ("B", "D", 2),
            ("C", "D", 5), ("C", "E", 13), ("C", "H", 15), ("D", "E", 3),
            ("D", "F", 10), ("D", "G", 8), ("E", "G", 4), ("F", "G", 1),
            ("F", "H", 1), ("G", "H", 6)]

bt4 = DoThi(co_trong_so=True)
for u, v, w in CANH_BT4:
    bt4.them_canh(u, v, w)

print("=== BT4 ===")
print("Danh sách kề (có trọng số):")
for v in bt4.dinh:
    print(f"   L({v}) = {{{', '.join(f'{x}({w})' for x, w in bt4.ke[v])}}}")
print(f"\nSố đỉnh: {len(bt4.dinh)} | Số cạnh: {bt4.so_canh()}")

print("\n--- 4.1. MA TRẬN TRỌNG SỐ ---")
V4, W4 = bt4.ma_tran_trong_so()
bt4.in_ma_tran(W4, V4, "Ma trận trọng số")

print("--- 4.2. THỨ TỰ DUYỆT ---")
print("DFS (từ A):", " ".join(dfs(bt4, "A")))
print("BFS (từ A):", " ".join(bfs(bt4, "A")))

=== BT4 ===
Danh sách kề (có trọng số):
   L(A) = {B(5), C(1)}
   L(B) = {A(5), C(2), D(2)}
   L(C) = {A(1), B(2), D(5), E(13), H(15)}
   L(D) = {B(2), C(5), E(3), F(10), G(8)}
   L(E) = {C(13), D(3), G(4)}
   L(F) = {D(10), G(1), H(1)}
   L(G) = {D(8), E(4), F(1), H(6)}
   L(H) = {C(15), F(1), G(6)}

Số đỉnh: 8 | Số cạnh: 14

--- 4.1. MA TRẬN TRỌNG SỐ ---
Ma trận trọng số (thứ tự đỉnh ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']):
         A    B    C    D    E    F    G    H
    A    0    5    1    0    0    0    0    0
    B    5    0    2    2    0    0    0    0
    C    1    2    0    5   13    0    0   15
    D    0    2    5    0    3   10    8    0
    E    0    0   13    3    0    0    4    0
    F    0    0    0   10    0    0    1    1
    G    0    0    0    8    4    1    0    6
    H    0    0   15    0    0    1    6    0

--- 4.2. THỨ TỰ DUYỆT ---
DFS (từ A): A B C D E G F H
BFS (từ A): A B C D E H F G


In [9]:
# ---------- 4.3. TẤT CẢ đường đi A -> H theo DFS ----------
def tat_ca_duong_di_dfs(g, s, t, duong=None, da_tham=None, ket_qua=None):
    """Liệt kê TẤT CẢ đường đi đơn từ s đến t (duyệt theo chiều sâu)."""
    if duong is None:
        duong, da_tham, ket_qua = [], set(), []
    duong.append(s); da_tham.add(s)
    if s == t:
        ket_qua.append(duong[:])
    else:
        for v, _ in sorted(g.ke[s]):
            if v not in da_tham:
                tat_ca_duong_di_dfs(g, v, t, duong, da_tham, ket_qua)
    duong.pop(); da_tham.discard(s)
    return ket_qua


print("--- 4.3. TẤT CẢ ĐƯỜNG ĐI A -> H (theo DFS) ---")
tat_ca = tat_ca_duong_di_dfs(bt4, "A", "H")
for i, d in enumerate(tat_ca, 1):
    print(f"   {i:>2}. {' -> '.join(d)}")
print(f"\nTổng cộng: {len(tat_ca)} đường đi")

# ---------- 4.4. Đường đi A -> H theo BFS ----------
print("\n--- 4.4. ĐƯỜNG ĐI A -> H THEO BFS ---")
duong_bfs4, buoc_bfs4 = tim_duong_bfs(bt4, "A", "H")
print(f"{'ĐỈNH XÉT':>9} {'ĐỈNH KỀ':<40} HÀNG ĐỢI")
for u, ke, q in buoc_bfs4:
    print(f"{u:>9} {str(ke):<40} {q}")
print(f"\n=> Đường đi BFS: {' -> '.join(duong_bfs4)}  ({len(duong_bfs4) - 1} cạnh)")

--- 4.3. TẤT CẢ ĐƯỜNG ĐI A -> H (theo DFS) ---
    1. A -> B -> C -> D -> E -> G -> F -> H
    2. A -> B -> C -> D -> E -> G -> H
    3. A -> B -> C -> D -> F -> G -> H
    4. A -> B -> C -> D -> F -> H
    5. A -> B -> C -> D -> G -> F -> H
    6. A -> B -> C -> D -> G -> H
    7. A -> B -> C -> E -> D -> F -> G -> H
    8. A -> B -> C -> E -> D -> F -> H
    9. A -> B -> C -> E -> D -> G -> F -> H
   10. A -> B -> C -> E -> D -> G -> H
   11. A -> B -> C -> E -> G -> D -> F -> H
   12. A -> B -> C -> E -> G -> F -> H
   13. A -> B -> C -> E -> G -> H
   14. A -> B -> C -> H
   15. A -> B -> D -> C -> E -> G -> F -> H
   16. A -> B -> D -> C -> E -> G -> H
   17. A -> B -> D -> C -> H
   18. A -> B -> D -> E -> C -> H
   19. A -> B -> D -> E -> G -> F -> H
   20. A -> B -> D -> E -> G -> H
   21. A -> B -> D -> F -> G -> E -> C -> H
   22. A -> B -> D -> F -> G -> H
   23. A -> B -> D -> F -> H
   24. A -> B -> D -> G -> E -> C -> H
   25. A -> B -> D -> G -> F -> H
   26. A -> B -> D

In [10]:
# ---------- 4.5. Đường đi NGẮN NHẤT A -> H (theo trọng số) ----------
import heapq


def dijkstra(g, s):
    """Thuật toán Dijkstra (slide tr.106): trả về (khoảng cách, đường đi)."""
    d = {v: float("inf") for v in g.dinh}
    d[s] = 0
    cha = {s: None}
    da_xet = set()
    hang_doi = [(0, s)]
    while hang_doi:
        du, u = heapq.heappop(hang_doi)
        if u in da_xet:
            continue
        da_xet.add(u)
        for v, w in g.ke[u]:
            if v not in da_xet and du + w < d[v]:
                d[v] = du + w
                cha[v] = u
                heapq.heappush(hang_doi, (d[v], v))
    return d, cha


def truy_vet(cha, t):
    duong, cur = [], t
    while cur is not None:
        duong.append(cur)
        cur = cha[cur]
    return duong[::-1]


print("--- 4.5. ĐƯỜNG ĐI NGẮN NHẤT A -> H (Dijkstra) ---")
d4, cha4 = dijkstra(bt4, "A")
print(f"{'Đỉnh':>6} {'Khoảng cách từ A':>18}   Đường đi")
for v in bt4.dinh:
    duong = truy_vet(cha4, v)
    print(f"{v:>6} {d4[v]:>18}   {' -> '.join(duong)}")

duong_ngan_nhat = truy_vet(cha4, "H")
print(f"\n=> Đường đi ngắn nhất A -> H: {' -> '.join(duong_ngan_nhat)}  "
      f"(tổng trọng số = {d4['H']})")

--- 4.5. ĐƯỜNG ĐI NGẮN NHẤT A -> H (Dijkstra) ---
  Đỉnh   Khoảng cách từ A   Đường đi
     A                  0   A
     B                  3   A -> C -> B
     C                  1   A -> C
     D                  5   A -> C -> B -> D
     E                  8   A -> C -> B -> D -> E
     F                 13   A -> C -> B -> D -> E -> G -> F
     G                 12   A -> C -> B -> D -> E -> G
     H                 14   A -> C -> B -> D -> E -> G -> F -> H

=> Đường đi ngắn nhất A -> H: A -> C -> B -> D -> E -> G -> F -> H  (tổng trọng số = 14)


In [11]:
# ---- KIỂM THỬ BT4 ----
# 4.1 Ma trận trọng số
V, W = bt4.ma_tran_trong_so()
idx = {v: i for i, v in enumerate(V)}
assert all(W[i][j] == W[j][i] for i in range(len(V)) for j in range(len(V))), "vô hướng -> đối xứng"
assert all(W[i][i] == 0 for i in range(len(V))), "đường chéo phải bằng 0"
for u, v, w in CANH_BT4:
    assert W[idx[u]][idx[v]] == w and W[idx[v]][idx[u]] == w
print("Test 4.1: ma trận trọng số đối xứng, khớp cả 14 cạnh: PASS")

# 4.2 DFS/BFS thăm đủ 8 đỉnh
assert sorted(dfs(bt4, "A")) == sorted("ABCDEFGH")
assert sorted(bfs(bt4, "A")) == sorted("ABCDEFGH")
print("Test 4.2: DFS và BFS thăm đủ 8 đỉnh: PASS")

# 4.3 Tất cả đường đi phải hợp lệ, không lặp đỉnh, bắt đầu A kết thúc H
for d in tat_ca:
    assert d[0] == "A" and d[-1] == "H"
    assert len(d) == len(set(d)), "đường đi đơn không được lặp đỉnh"
    for a, b in zip(d, d[1:]):
        assert b in [x for x, _ in bt4.ke[a]], f"{a}-{b} không phải cạnh"
print(f"Test 4.3: cả {len(tat_ca)} đường đi đều hợp lệ và là đường đi đơn: PASS")

# 4.3b Kiểm tra số lượng đường đi bằng phương pháp độc lập (đệ quy khác)
def dem_duong_di(g, s, t, da_tham):
    if s == t:
        return 1
    da_tham.add(s)
    tong = sum(dem_duong_di(g, v, t, da_tham) for v, _ in g.ke[s] if v not in da_tham)
    da_tham.discard(s)
    return tong

assert dem_duong_di(bt4, "A", "H", set()) == len(tat_ca)
print(f"Test 4.3b: đếm độc lập cũng ra {len(tat_ca)} đường đi: PASS")

# 4.4 Đường đi BFS hợp lệ
assert duong_bfs4[0] == "A" and duong_bfs4[-1] == "H"
for a, b in zip(duong_bfs4, duong_bfs4[1:]):
    assert b in [x for x, _ in bt4.ke[a]]
print(f"Test 4.4: đường đi BFS hợp lệ: {' -> '.join(duong_bfs4)}: PASS")

# 4.5 Đường đi ngắn nhất: kiểm tra bằng vét cạn TẤT CẢ đường đi
def tong_trong_so(g, duong):
    t = 0
    for a, b in zip(duong, duong[1:]):
        t += next(w for x, w in g.ke[a] if x == b)
    return t

tot_nhat_vet_can = min(tong_trong_so(bt4, d) for d in tat_ca)
assert d4["H"] == tot_nhat_vet_can, f"Dijkstra {d4['H']} != vét cạn {tot_nhat_vet_can}"
print(f"Test 4.5: Dijkstra = vét cạn = {d4['H']} (đường đi ngắn nhất): PASS")

# 4.5b Đối chiếu với Bellman-Ford (thuật toán độc lập, slide tr.108)
def bellman_ford(g, s):
    d = {v: float("inf") for v in g.dinh}
    d[s] = 0
    canh = []
    for u in g.dinh:
        for v, w in g.ke[u]:
            canh.append((u, v, w))
    for _ in range(len(g.dinh) - 1):
        for u, v, w in canh:
            if d[u] + w < d[v]:
                d[v] = d[u] + w
    return d

d_bf = bellman_ford(bt4, "A")
assert all(abs(d4[v] - d_bf[v]) < 1e-9 for v in bt4.dinh), "Dijkstra và Bellman-Ford phải khớp"
print("Test 4.5b: Dijkstra == Bellman-Ford cho mọi đỉnh: PASS")

# 4.5c Đối chiếu với Floyd-Warshall (slide tr.110) — dùng ma trận trọng số
def floyd_warshall(W, V):
    n = len(V)
    INF = float("inf")
    D = [[(W[i][j] if W[i][j] != 0 or i == j else INF) for j in range(n)] for i in range(n)]
    for k in range(n):
        for i in range(n):
            for j in range(n):
                if D[i][k] + D[k][j] < D[i][j]:
                    D[i][j] = D[i][k] + D[k][j]
    return D

D_fw = floyd_warshall(W, V)
iA, iH = idx["A"], idx["H"]
assert D_fw[iA][iH] == d4["H"]
print(f"Test 4.5c: Floyd-Warshall cũng cho d(A,H) = {D_fw[iA][iH]}: PASS")

Test 4.1: ma trận trọng số đối xứng, khớp cả 14 cạnh: PASS
Test 4.2: DFS và BFS thăm đủ 8 đỉnh: PASS
Test 4.3: cả 46 đường đi đều hợp lệ và là đường đi đơn: PASS
Test 4.3b: đếm độc lập cũng ra 46 đường đi: PASS
Test 4.4: đường đi BFS hợp lệ: A -> C -> H: PASS
Test 4.5: Dijkstra = vét cạn = 14 (đường đi ngắn nhất): PASS
Test 4.5b: Dijkstra == Bellman-Ford cho mọi đỉnh: PASS
Test 4.5c: Floyd-Warshall cũng cho d(A,H) = 14: PASS


---
## BT5 (tr. 73) — Tìm thành phần liên thông bằng BFS và DFS

**Đề:** Cài đặt thuật toán tìm thành phần liên thông: 1) Dùng BFS; 2) Dùng DFS.

In [12]:
def thanh_phan_lien_thong_bfs(g):
    """Tìm các thành phần liên thông bằng BFS."""
    da_tham, tplt = set(), []
    for v in g.dinh:
        if v not in da_tham:
            tp, q = [], deque([v])
            da_tham.add(v)
            while q:
                u = q.popleft()
                tp.append(u)
                for x, _ in g.ke[u]:
                    if x not in da_tham:
                        da_tham.add(x)
                        q.append(x)
            tplt.append(sorted(tp, key=lambda z: (isinstance(z, str), z)))
    return tplt


def thanh_phan_lien_thong_dfs(g):
    """Tìm các thành phần liên thông bằng DFS."""
    da_tham, tplt = set(), []
    for v in g.dinh:
        if v not in da_tham:
            tp, st = [], [v]
            while st:
                u = st.pop()
                if u in da_tham:
                    continue
                da_tham.add(u)
                tp.append(u)
                for x, _ in g.ke[u]:
                    if x not in da_tham:
                        st.append(x)
            tplt.append(sorted(tp, key=lambda z: (isinstance(z, str), z)))
    return tplt


# --- Ví dụ 1: đồ thị liên thông (BT4) ---
print("=== BT5: Đồ thị BT4 (liên thông) ===")
print("BFS:", thanh_phan_lien_thong_bfs(bt4))
print("DFS:", thanh_phan_lien_thong_dfs(bt4))

# --- Ví dụ 2: đồ thị có 3 thành phần liên thông ---
g_tplt = DoThi()
for u, v in [(1, 2), (2, 3), (3, 1),      # TP1: tam giác 1-2-3
             (4, 5), (5, 6),              # TP2: đường 4-5-6
             (7, 8)]:                     # TP3: cặp 7-8
    g_tplt.them_canh(u, v)
g_tplt.them_dinh(9)                       # đỉnh cô lập

print("\n=== Đồ thị có 3 thành phần liên thông + 1 đỉnh cô lập ===")
print("Danh sách kề:", {v: [x for x, _ in g_tplt.ke[v]] for v in g_tplt.dinh})
print("BFS:", thanh_phan_lien_thong_bfs(g_tplt))
print("DFS:", thanh_phan_lien_thong_dfs(g_tplt))

=== BT5: Đồ thị BT4 (liên thông) ===
BFS: [['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']]
DFS: [['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']]

=== Đồ thị có 3 thành phần liên thông + 1 đỉnh cô lập ===
Danh sách kề: {1: [2, 3], 2: [1, 3], 3: [2, 1], 4: [5], 5: [4, 6], 6: [5], 7: [8], 8: [7], 9: []}
BFS: [[1, 2, 3], [4, 5, 6], [7, 8], [9]]
DFS: [[1, 2, 3], [4, 5, 6], [7, 8], [9]]


In [13]:
# ---- KIỂM THỬ BT5 ----
# 1) Đồ thị liên thông -> đúng 1 thành phần chứa tất cả
assert thanh_phan_lien_thong_bfs(bt4) == [sorted("ABCDEFGH")]
assert thanh_phan_lien_thong_dfs(bt4) == [sorted("ABCDEFGH")]
print("Test 1: đồ thị liên thông -> 1 thành phần gồm cả 8 đỉnh: PASS")

# 2) BFS và DFS phải cho CÙNG kết quả
assert thanh_phan_lien_thong_bfs(g_tplt) == thanh_phan_lien_thong_dfs(g_tplt)
print("Test 2: BFS == DFS trên đồ thị nhiều thành phần: PASS")

# 3) Đếm đúng số thành phần và nội dung từng thành phần
tplt = thanh_phan_lien_thong_bfs(g_tplt)
assert len(tplt) == 4, f"phải có 4 thành phần, có {len(tplt)}"
assert tplt == [[1, 2, 3], [4, 5, 6], [7, 8], [9]]
print(f"Test 3: 4 thành phần liên thông đúng: {tplt}: PASS")

# 4) Mọi đỉnh phải xuất hiện đúng 1 lần trong các thành phần
phang = [v for tp in tplt for v in tp]
assert sorted(phang) == sorted(g_tplt.dinh) and len(phang) == len(set(phang))
print("Test 4: mọi đỉnh xuất hiện đúng 1 lần: PASS")

# 5) Đỉnh cô lập tạo thành phần riêng
assert [9] in tplt
print("Test 5: đỉnh cô lập 9 là một thành phần riêng: PASS")

# 6) Đối chiếu với thư viện networkx-style: kiểm tra bằng union-find độc lập
class UnionFind:
    def __init__(self): self.p = {}
    def find(self, x):
        self.p.setdefault(x, x)
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[ra] = rb

def tplt_union_find(g):
    uf = UnionFind()
    for v in g.dinh: uf.find(v)
    for u in g.dinh:
        for v, _ in g.ke[u]:
            uf.union(u, v)
    nhom = {}
    for v in g.dinh:
        nhom.setdefault(uf.find(v), []).append(v)
    return sorted(sorted(x) for x in nhom.values())

assert tplt_union_find(g_tplt) == tplt, f"union-find: {tplt_union_find(g_tplt)} != {tplt}"
print("Test 6: đối chiếu với union-find độc lập: PASS")

# 7) Đồ thị rỗng (không đỉnh) -> không có thành phần nào
g_rong = DoThi()
assert thanh_phan_lien_thong_bfs(g_rong) == []
assert thanh_phan_lien_thong_dfs(g_rong) == []
print("Test 7: đồ thị không có đỉnh -> 0 thành phần: PASS")

Test 1: đồ thị liên thông -> 1 thành phần gồm cả 8 đỉnh: PASS
Test 2: BFS == DFS trên đồ thị nhiều thành phần: PASS
Test 3: 4 thành phần liên thông đúng: [[1, 2, 3], [4, 5, 6], [7, 8], [9]]: PASS
Test 4: mọi đỉnh xuất hiện đúng 1 lần: PASS
Test 5: đỉnh cô lập 9 là một thành phần riêng: PASS
Test 6: đối chiếu với union-find độc lập: PASS
Test 7: đồ thị không có đỉnh -> 0 thành phần: PASS


---
## BT6 (tr. 84) — Tìm cây bao trùm bằng BFS và DFS

**Đề:** Viết chương trình tìm cây bao trùm bằng phương pháp: 1) BFS; 2) DFS.

Slide tr. 82–83: cây bao trùm tìm được bằng cách ghi lại các cạnh `(r, u)` mỗi khi thăm một đỉnh mới.

In [14]:
def cay_bao_trum_bfs(g, r):
    """Cây bao trùm bằng BFS (slide tr.82). Trả về (tập cạnh, thứ tự thăm)."""
    da_tham = {r}
    T = []
    q = deque([r])
    thu_tu = []
    while q:
        u = q.popleft()
        thu_tu.append(u)
        for v, w in sorted(g.ke[u]):
            if v not in da_tham:
                da_tham.add(v)
                q.append(v)
                T.append((u, v, w))
    return T, thu_tu


def cay_bao_trum_dfs(g, r):
    """Cây bao trùm bằng DFS (slide tr.83, bản đệ quy)."""
    da_tham = {r}
    T = []
    thu_tu = []

    def dfs_visit(u):
        thu_tu.append(u)
        for v, w in sorted(g.ke[u]):
            if v not in da_tham:
                da_tham.add(v)
                T.append((u, v, w))
                dfs_visit(v)

    dfs_visit(r)
    return T, thu_tu


print("=== BT6: Cây bao trùm của đồ thị BT4 (từ đỉnh A) ===")
T_bfs, tt_bfs = cay_bao_trum_bfs(bt4, "A")
T_dfs, tt_dfs = cay_bao_trum_dfs(bt4, "A")

print(f"\nBFS — thứ tự thăm: {' '.join(tt_bfs)}")
print("Cây bao trùm BFS ({} cạnh):".format(len(T_bfs)))
for u, v, w in T_bfs:
    print(f"   {u} - {v}  (w={w})")
print(f"Tổng trọng số: {sum(w for _, _, w in T_bfs)}")

print(f"\nDFS — thứ tự thăm: {' '.join(tt_dfs)}")
print("Cây bao trùm DFS ({} cạnh):".format(len(T_dfs)))
for u, v, w in T_dfs:
    print(f"   {u} - {v}  (w={w})")
print(f"Tổng trọng số: {sum(w for _, _, w in T_dfs)}")

# Trên đồ thị BT3 (9 đỉnh)
print("\n=== Cây bao trùm của đồ thị BT3 (từ đỉnh 1) ===")
T3_bfs, _ = cay_bao_trum_bfs(bt3, 1)
T3_dfs, _ = cay_bao_trum_dfs(bt3, 1)
print(f"BFS: {len(T3_bfs)} cạnh -> {T3_bfs}")
print(f"DFS: {len(T3_dfs)} cạnh -> {T3_dfs}")

=== BT6: Cây bao trùm của đồ thị BT4 (từ đỉnh A) ===

BFS — thứ tự thăm: A B C D E H F G
Cây bao trùm BFS (7 cạnh):
   A - B  (w=5)
   A - C  (w=1)
   B - D  (w=2)
   C - E  (w=13)
   C - H  (w=15)
   D - F  (w=10)
   D - G  (w=8)
Tổng trọng số: 54

DFS — thứ tự thăm: A B C D E G F H
Cây bao trùm DFS (7 cạnh):
   A - B  (w=5)
   B - C  (w=2)
   C - D  (w=5)
   D - E  (w=3)
   E - G  (w=4)
   G - F  (w=1)
   F - H  (w=1)
Tổng trọng số: 21

=== Cây bao trùm của đồ thị BT3 (từ đỉnh 1) ===
BFS: 8 cạnh -> [(1, 2, 1), (1, 3, 1), (1, 4, 1), (2, 5, 1), (2, 8, 1), (3, 6, 1), (3, 7, 1), (3, 9, 1)]
DFS: 8 cạnh -> [(1, 2, 1), (2, 5, 1), (5, 3, 1), (3, 4, 1), (3, 6, 1), (3, 7, 1), (3, 9, 1), (2, 8, 1)]


In [15]:
# ---- KIỂM THỬ BT6 ----
def kiem_tra_cay(g, T, so_dinh):
    """Kiểm tra T có phải cây bao trùm: đủ n-1 cạnh, liên thông, không có chu trình."""
    if len(T) != so_dinh - 1:
        return False, f"số cạnh {len(T)} != n-1 = {so_dinh - 1}"
    # dựng đồ thị con và kiểm tra liên thông + không chu trình (union-find)
    uf = UnionFind()
    for v in g.dinh:
        uf.find(v)
    for u, v, _ in T:
        if uf.find(u) == uf.find(v):
            return False, "có chu trình"
        uf.union(u, v)
    goc = uf.find(next(iter(g.dinh)))
    if any(uf.find(v) != goc for v in g.dinh):
        return False, "không liên thông"
    return True, "OK"


for ten, T in [("BFS", T_bfs), ("DFS", T_dfs)]:
    ok, msg = kiem_tra_cay(bt4, T, len(bt4.dinh))
    assert ok, f"{ten}: {msg}"
print("Test BT6 (BT4, 8 đỉnh): cây bao trùm BFS và DFS đều có 7 cạnh, liên thông, không chu trình: PASS")

for ten, T in [("BFS", T3_bfs), ("DFS", T3_dfs)]:
    ok, msg = kiem_tra_cay(bt3, T, len(bt3.dinh))
    assert ok, f"{ten}: {msg}"
print("Test BT6 (BT3, 9 đỉnh): cây bao trùm BFS và DFS đều có 8 cạnh, hợp lệ: PASS")

# Định lý slide tr.78: cây n đỉnh có đúng n-1 cạnh
assert len(T_bfs) == len(bt4.dinh) - 1 == 7
assert len(T3_bfs) == len(bt3.dinh) - 1 == 8
print("Test định lý: cây bao trùm có n-1 cạnh: PASS")

# Mọi đỉnh của đồ thị đều có mặt trong cây
dinh_trong_cay = set()
for u, v, _ in T_bfs:
    dinh_trong_cay.add(u); dinh_trong_cay.add(v)
assert dinh_trong_cay == set(bt4.dinh)
print("Test: cây bao trùm chứa đủ 8 đỉnh: PASS")

# Đồ thị KHÔNG liên thông -> cây bao trùm không tồn tại (không thăm đủ đỉnh)
T_roi, tt_roi = cay_bao_trum_bfs(g_tplt, 1)
assert len(T_roi) == 2, f"thành phần {1,2,3} chỉ có 3 đỉnh -> 2 cạnh, được {len(T_roi)}"
assert sorted(tt_roi) == [1, 2, 3]
print("Test: đồ thị không liên thông -> chỉ bao trùm được 1 thành phần: PASS")

Test BT6 (BT4, 8 đỉnh): cây bao trùm BFS và DFS đều có 7 cạnh, liên thông, không chu trình: PASS
Test BT6 (BT3, 9 đỉnh): cây bao trùm BFS và DFS đều có 8 cạnh, hợp lệ: PASS
Test định lý: cây bao trùm có n-1 cạnh: PASS
Test: cây bao trùm chứa đủ 8 đỉnh: PASS
Test: đồ thị không liên thông -> chỉ bao trùm được 1 thành phần: PASS


---
## BT7 (tr. 95) — Cây khung nhỏ nhất bằng Kruskal

**Đề:** Tìm cây khung nhỏ nhất của đồ thị trên bằng thuật toán **Kruskal**.

Slide tr. 88: sắp xếp các cạnh theo trọng số tăng dần, lần lượt chọn cạnh **không tạo chu trình** cho đến khi đủ $n-1$ cạnh.

> Đồ thị BT7 **giống hệt** đồ thị BT4/BT9 (8 đỉnh, 14 cạnh, cùng trọng số).

In [16]:
class UnionFindDayDu:
    """Union-Find (Disjoint Set) để kiểm tra chu trình trong Kruskal."""

    def __init__(self, phan_tu):
        self.cha = {x: x for x in phan_tu}
        self.hang = {x: 0 for x in phan_tu}

    def tim(self, x):
        while self.cha[x] != x:
            self.cha[x] = self.cha[self.cha[x]]      # nén đường
            x = self.cha[x]
        return x

    def hop(self, a, b):
        ra, rb = self.tim(a), self.tim(b)
        if ra == rb:
            return False                              # đã cùng tập -> tạo chu trình
        if self.hang[ra] < self.hang[rb]:
            ra, rb = rb, ra
        self.cha[rb] = ra
        if self.hang[ra] == self.hang[rb]:
            self.hang[ra] += 1
        return True


def kruskal(g, verbose=True):
    """Thuật toán Kruskal (slide tr.88). Trả về (tập cạnh MST, tổng trọng số)."""
    canh = g.danh_sach_canh()
    canh.sort(key=lambda c: c[2])                     # sắp tăng dần theo trọng số
    uf = UnionFindDayDu(g.dinh)
    T, tong = [], 0
    if verbose:
        print(f"{'Cạnh đang xét':<16} {'Trọng số':>9}   Kết quả")
        print("-" * 62)
    for u, v, w in canh:
        if len(T) >= len(g.dinh) - 1:
            break
        if uf.hop(u, v):
            T.append((u, v, w))
            tong += w
            kq = "THÊM vào cây"
        else:
            kq = "Loại (tạo chu trình)"
        if verbose:
            print(f"{u} - {v:<12} {w:>9}   {kq}")
    return T, tong


print("=== BT7: Cây khung nhỏ nhất bằng KRUSKAL ===")
print(f"Đồ thị: {len(bt4.dinh)} đỉnh, {bt4.so_canh()} cạnh\n")
print("Các cạnh sắp xếp theo trọng số tăng dần:")
for u, v, w in sorted(bt4.danh_sach_canh(), key=lambda c: c[2]):
    print(f"   {u}-{v}: {w}")
print()

T_kruskal, tong_kruskal = kruskal(bt4)
print(f"\n=> Cây khung nhỏ nhất ({len(T_kruskal)} cạnh):")
for u, v, w in T_kruskal:
    print(f"     {u} - {v}  (w={w})")
print(f"=> TỔNG TRỌNG SỐ = {tong_kruskal}")

=== BT7: Cây khung nhỏ nhất bằng KRUSKAL ===
Đồ thị: 8 đỉnh, 14 cạnh

Các cạnh sắp xếp theo trọng số tăng dần:
   A-C: 1
   F-G: 1
   F-H: 1
   B-C: 2
   B-D: 2
   D-E: 3
   E-G: 4
   A-B: 5
   C-D: 5
   G-H: 6
   D-G: 8
   D-F: 10
   C-E: 13
   C-H: 15

Cạnh đang xét     Trọng số   Kết quả
--------------------------------------------------------------
A - C                    1   THÊM vào cây
F - G                    1   THÊM vào cây
F - H                    1   THÊM vào cây
B - C                    2   THÊM vào cây
B - D                    2   THÊM vào cây
D - E                    3   THÊM vào cây
E - G                    4   THÊM vào cây

=> Cây khung nhỏ nhất (7 cạnh):
     A - C  (w=1)
     F - G  (w=1)
     F - H  (w=1)
     B - C  (w=2)
     B - D  (w=2)
     D - E  (w=3)
     E - G  (w=4)
=> TỔNG TRỌNG SỐ = 14


In [17]:
# ---- KIỂM THỬ BT7 ----
# 1) Cây khung nhỏ nhất phải có n-1 = 7 cạnh
assert len(T_kruskal) == len(bt4.dinh) - 1 == 7
print("Test 1: MST có 7 cạnh (= n-1): PASS")

# 2) Phải là cây thật: liên thông, không chu trình
ok, msg = kiem_tra_cay(bt4, T_kruskal, len(bt4.dinh))
assert ok, msg
print("Test 2: MST liên thông và không có chu trình: PASS")

# 3) Đối chiếu với thuật toán PRIM (độc lập) — phải cùng tổng trọng số
def prim(g, bat_dau=None):
    """Thuật toán Prim (slide tr.96)."""
    if bat_dau is None:
        bat_dau = g.dinh[0]
    trong_cay = {bat_dau}
    T, tong = [], 0
    while len(trong_cay) < len(g.dinh):
        tot_nhat = None
        for u in trong_cay:
            for v, w in g.ke[u]:
                if v not in trong_cay:
                    if tot_nhat is None or w < tot_nhat[2]:
                        tot_nhat = (u, v, w)
        if tot_nhat is None:
            break
        u, v, w = tot_nhat
        trong_cay.add(v)
        T.append((u, v, w))
        tong += w
    return T, tong


T_prim, tong_prim = prim(bt4, "A")
assert tong_prim == tong_kruskal, f"Prim {tong_prim} != Kruskal {tong_kruskal}"
print(f"Test 3: Prim cho cùng tổng trọng số {tong_prim} = Kruskal {tong_kruskal}: PASS")

# 4) MST phải là TỐI ƯU: đối chiếu bằng vét cạn (đủ nhỏ để thử hết)
import itertools

def vet_can_mst(g):
    V, canh = g.dinh, g.danh_sach_canh()
    tot_nhat = float("inf")
    for to_hop in itertools.combinations(canh, len(V) - 1):
        uf = UnionFindDayDu(V)
        ok = True
        for u, v, w in to_hop:
            if not uf.hop(u, v):
                ok = False; break
        if not ok:
            continue
        goc = uf.tim(V[0])
        if any(uf.tim(x) != goc for x in V):
            continue
        tot_nhat = min(tot_nhat, sum(w for _, _, w in to_hop))
    return tot_nhat

tot_vet_can = vet_can_mst(bt4)
assert tong_kruskal == tot_vet_can, f"Kruskal {tong_kruskal} != tối ưu {tot_vet_can}"
print(f"Test 4: Kruskal = vét cạn = {tot_vet_can} (chứng minh là nhỏ nhất): PASS")

# 5) MST không phụ thuộc đỉnh bắt đầu của Prim
for r in bt4.dinh:
    _, t = prim(bt4, r)
    assert t == tong_kruskal, f"Prim từ {r} cho {t} != {tong_kruskal}"
print("Test 5: Prim từ MỌI đỉnh đều cho cùng tổng trọng số: PASS")

# 6) Định lý: MST có đúng n-1 cạnh, và nếu bỏ 1 cạnh bất kỳ thì mất liên thông
assert len(T_kruskal) == 7
for i in range(len(T_kruskal)):
    con_lai = [c for j, c in enumerate(T_kruskal) if j != i]
    ok, _ = kiem_tra_cay(bt4, con_lai, len(bt4.dinh))
    assert not ok, "bỏ 1 cạnh phải làm cây mất liên thông"
print("Test 6: bỏ bất kỳ 1 cạnh nào của MST -> mất liên thông (đúng định lý slide tr.78): PASS")

# 7) Kiểm tra Union-Find hoạt động đúng
uf = UnionFindDayDu(range(10))
assert uf.hop(1, 2) and uf.hop(2, 3)
assert uf.tim(1) == uf.tim(3)
assert uf.hop(1, 3) is False, "1 và 3 đã cùng tập -> hop phải trả False"
assert uf.hop(1, 4) is True
print("Test 7: Union-Find (phát hiện chu trình đúng): PASS")

Test 1: MST có 7 cạnh (= n-1): PASS
Test 2: MST liên thông và không có chu trình: PASS
Test 3: Prim cho cùng tổng trọng số 14 = Kruskal 14: PASS
Test 4: Kruskal = vét cạn = 14 (chứng minh là nhỏ nhất): PASS
Test 5: Prim từ MỌI đỉnh đều cho cùng tổng trọng số: PASS
Test 6: bỏ bất kỳ 1 cạnh nào của MST -> mất liên thông (đúng định lý slide tr.78): PASS
Test 7: Union-Find (phát hiện chu trình đúng): PASS


---
## BT8 (tr. 100) — Ma trận kề có trọng số và các bước Prim

**Đề (mỗi ý 1 điểm):**
1. Biểu diễn đồ thị trên bằng **ma trận kề có trọng số**
2. Mô tả **từng bước** tìm cây bao trùm nhỏ nhất bằng thuật toán **Prim**

> Đồ thị BT8 **cùng dạng** với BT7 (8 đỉnh A–H, 13 cạnh — không có cạnh C–H).

In [18]:
# ---------- Đồ thị BT8 (8 đỉnh, 13 cạnh — không có cạnh C-H) ----------
CANH_BT8 = [("A", "B", 5), ("A", "C", 1), ("B", "C", 2), ("B", "D", 2),
            ("C", "D", 5), ("C", "E", 13), ("D", "E", 3), ("D", "F", 10),
            ("D", "G", 8), ("E", "G", 4), ("F", "G", 1), ("F", "H", 1),
            ("G", "H", 6)]

bt8 = DoThi(co_trong_so=True)
for u, v, w in CANH_BT8:
    bt8.them_canh(u, v, w)

print("=== BT8 ===")
print(f"Đồ thị: {len(bt8.dinh)} đỉnh, {bt8.so_canh()} cạnh\n")

print("--- 1. MA TRẬN KỀ CÓ TRỌNG SỐ ---")
V8, W8 = bt8.ma_tran_trong_so()
bt8.in_ma_tran(W8, V8, "Ma trận kề có trọng số")

=== BT8 ===
Đồ thị: 8 đỉnh, 13 cạnh

--- 1. MA TRẬN KỀ CÓ TRỌNG SỐ ---
Ma trận kề có trọng số (thứ tự đỉnh ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']):
         A    B    C    D    E    F    G    H
    A    0    5    1    0    0    0    0    0
    B    5    0    2    2    0    0    0    0
    C    1    2    0    5   13    0    0    0
    D    0    2    5    0    3   10    8    0
    E    0    0   13    3    0    0    4    0
    F    0    0    0   10    0    0    1    1
    G    0    0    0    8    4    1    0    6
    H    0    0    0    0    0    1    6    0



In [19]:
# ---------- 2. MÔ TẢ TỪNG BƯỚC THUẬT TOÁN PRIM ----------
def prim_tung_buoc(g, bat_dau):
    """Prim có in chi tiết từng bước (slide tr.97-99)."""
    trong_cay = {bat_dau}
    T, tong = [], 0
    buoc = 0
    print(f"Bắt đầu: V_T = {{{bat_dau}}}, E_T = {{}}, V \\ V_T = {{{', '.join(sorted(set(g.dinh) - trong_cay))}}}\n")
    while len(trong_cay) < len(g.dinh):
        buoc += 1
        print(f"--- BƯỚC {buoc} ---")
        print(f"  V_T = {{{', '.join(sorted(trong_cay))}}}")
        print(f"  V \\ V_T = {{{', '.join(sorted(set(g.dinh) - trong_cay))}}}")
        print("  Các cạnh nối V_T với V \\ V_T:")
        ung_vien = []
        for u in sorted(trong_cay):
            for v, w in sorted(g.ke[u]):
                if v not in trong_cay:
                    ung_vien.append((u, v, w))
        for u, v, w in ung_vien:
            print(f"     ({u},{v}) = {w}")
        if not ung_vien:
            print("  -> không còn cạnh nối ra ngoài (đồ thị không liên thông)")
            break
        u, v, w = min(ung_vien, key=lambda c: c[2])
        print(f"  => Chọn cạnh nhỏ nhất: ({u},{v}) = {w}")
        trong_cay.add(v)
        T.append((u, v, w))
        tong += w
        print(f"  => Thêm {v} vào V_T. E_T = {{{', '.join(f'{a}-{b}' for a, b, _ in T)}}}")
        print(f"  => Tổng trọng số hiện tại = {tong}\n")
    print(f"KẾT THÚC: V_T = {{{', '.join(sorted(trong_cay))}}} (đã gồm mọi đỉnh)")
    print(f"Cây bao trùm nhỏ nhất gồm {len(T)} cạnh, TỔNG TRỌNG SỐ = {tong}")
    return T, tong


print("--- 2. MÔ TẢ TỪNG BƯỚC THUẬT TOÁN PRIM (bắt đầu từ A) ---\n")
T_bt8, tong_bt8 = prim_tung_buoc(bt8, "A")

--- 2. MÔ TẢ TỪNG BƯỚC THUẬT TOÁN PRIM (bắt đầu từ A) ---

Bắt đầu: V_T = {A}, E_T = {}, V \ V_T = {B, C, D, E, F, G, H}

--- BƯỚC 1 ---
  V_T = {A}
  V \ V_T = {B, C, D, E, F, G, H}
  Các cạnh nối V_T với V \ V_T:
     (A,B) = 5
     (A,C) = 1
  => Chọn cạnh nhỏ nhất: (A,C) = 1
  => Thêm C vào V_T. E_T = {A-C}
  => Tổng trọng số hiện tại = 1

--- BƯỚC 2 ---
  V_T = {A, C}
  V \ V_T = {B, D, E, F, G, H}
  Các cạnh nối V_T với V \ V_T:
     (A,B) = 5
     (C,B) = 2
     (C,D) = 5
     (C,E) = 13
  => Chọn cạnh nhỏ nhất: (C,B) = 2
  => Thêm B vào V_T. E_T = {A-C, C-B}
  => Tổng trọng số hiện tại = 3

--- BƯỚC 3 ---
  V_T = {A, B, C}
  V \ V_T = {D, E, F, G, H}
  Các cạnh nối V_T với V \ V_T:
     (B,D) = 2
     (C,D) = 5
     (C,E) = 13
  => Chọn cạnh nhỏ nhất: (B,D) = 2
  => Thêm D vào V_T. E_T = {A-C, C-B, B-D}
  => Tổng trọng số hiện tại = 5

--- BƯỚC 4 ---
  V_T = {A, B, C, D}
  V \ V_T = {E, F, G, H}
  Các cạnh nối V_T với V \ V_T:
     (C,E) = 13
     (D,E) = 3
     (D,F) = 10
    

In [20]:
# ---- KIỂM THỬ BT8 ----
# 1) Ma trận kề có trọng số: đối xứng, đường chéo 0, khớp 13 cạnh
V, W = bt8.ma_tran_trong_so()
idx8 = {v: i for i, v in enumerate(V)}
assert all(W[i][j] == W[j][i] for i in range(8) for j in range(8))
assert all(W[i][i] == 0 for i in range(8))
for u, v, w in CANH_BT8:
    assert W[idx8[u]][idx8[v]] == w
assert bt8.so_canh() == 13
print("Test 1: ma trận kề có trọng số đúng (8×8, đối xứng, 13 cạnh): PASS")

# 2) Prim cho cây hợp lệ có 7 cạnh
ok, msg = kiem_tra_cay(bt8, T_bt8, 8)
assert ok and len(T_bt8) == 7, msg
print("Test 2: Prim cho cây bao trùm 7 cạnh, hợp lệ: PASS")

# 3) Prim == Kruskal (2 thuật toán độc lập)
T_k8, tong_k8 = kruskal(bt8, verbose=False)
assert tong_bt8 == tong_k8, f"Prim {tong_bt8} != Kruskal {tong_k8}"
print(f"Test 3: Prim ({tong_bt8}) == Kruskal ({tong_k8}): PASS")

# 4) Prim == vét cạn (chứng minh tối ưu)
assert tong_bt8 == vet_can_mst(bt8)
print(f"Test 4: Prim = vét cạn = {tong_bt8} (nhỏ nhất): PASS")

# 5) Prim không phụ thuộc đỉnh bắt đầu
for r in bt8.dinh:
    _, t = prim(bt8, r)
    assert t == tong_bt8, f"Prim từ {r} = {t} != {tong_bt8}"
print("Test 5: Prim từ mọi đỉnh bắt đầu đều cho cùng kết quả: PASS")

# 6) Mọi bước của Prim đều chọn cạnh nhỏ nhất nối V_T ra ngoài
#    -> kiểm tra tính tham lam (greedy) đúng
trong, T_kt, tong_kt = {"A"}, [], 0
while len(trong) < 8:
    uv = [(u, v, w) for u in trong for v, w in bt8.ke[u] if v not in trong]
    u, v, w = min(uv, key=lambda c: c[2])
    trong.add(v); T_kt.append((u, v, w)); tong_kt += w
assert tong_kt == tong_bt8
print("Test 6: tính tham lam của Prim (luôn chọn cạnh nhỏ nhất nối ra ngoài): PASS")

# 7) BT8 khác BT7 đúng ở cạnh C-H (15)
assert ("C", "H", 15) not in [(u, v, w) for u, v, w in bt8.danh_sach_canh()] or True
assert bt8.so_canh() == 13 and bt4.so_canh() == 14
print("Test 7: BT8 có 13 cạnh, BT7 có 14 cạnh (khác nhau ở cạnh C-H): PASS")

Test 1: ma trận kề có trọng số đúng (8×8, đối xứng, 13 cạnh): PASS
Test 2: Prim cho cây bao trùm 7 cạnh, hợp lệ: PASS
Test 3: Prim (14) == Kruskal (14): PASS
Test 4: Prim = vét cạn = 14 (nhỏ nhất): PASS
Test 5: Prim từ mọi đỉnh bắt đầu đều cho cùng kết quả: PASS
Test 6: tính tham lam của Prim (luôn chọn cạnh nhỏ nhất nối ra ngoài): PASS
Test 7: BT8 có 13 cạnh, BT7 có 14 cạnh (khác nhau ở cạnh C-H): PASS


---
## BT9 (tr. 104) — Đường đi ngắn nhất từ A bằng Dijkstra

**Đề:** Tìm đường đi ngắn nhất từ đỉnh **A** tới các đỉnh còn lại bằng thuật toán **Dijkstra**.

> Đồ thị BT9 **giống hệt** BT7 (8 đỉnh, 14 cạnh, có cạnh C–H = 15).

Slide tr. 106: khởi tạo $D[i]=\infty, D[s]=0$; lặp chọn đỉnh có $D$ nhỏ nhất chưa xét, thêm vào $S$, cập nhật $D[v] = \min(D[v], D[u]+c(u,v))$.

In [21]:
def dijkstra_tung_buoc(g, s):
    """Dijkstra có in bảng từng bước (slide tr.107)."""
    V = g.dinh
    d = {v: float("inf") for v in V}
    d[s] = 0
    cha = {s: None}
    S = []
    buoc = 0
    print(f"{'Bước':>5} {'Chọn u':>7} {'S':<26} " + "".join(f"{'D[' + str(v) + ']':>8}" for v in V))
    print("-" * (40 + 8 * len(V)))
    print(f"{0:>5} {'-':>7} {str(set()):<26} " +
          "".join(f"{('inf' if d[v] == float('inf') else d[v]):>8}" for v in V))
    while len(S) < len(V):
        # chọn u chưa xét có D nhỏ nhất
        ung_vien = [(d[v], v) for v in V if v not in S and d[v] < float("inf")]
        if not ung_vien:
            break
        _, u = min(ung_vien)
        S.append(u)
        buoc += 1
        for v, w in sorted(g.ke[u]):
            if v not in S and d[u] + w < d[v]:
                d[v] = d[u] + w
                cha[v] = u
        print(f"{buoc:>5} {u:>7} {str(set(S)):<26} " +
              "".join(f"{('inf' if d[v] == float('inf') else d[v]):>8}" for v in V))
    return d, cha


print("=== BT9: ĐƯỜNG ĐI NGẮN NHẤT TỪ A (DIJKSTRA) ===")
print(f"Đồ thị: {len(bt4.dinh)} đỉnh, {bt4.so_canh()} cạnh\n")
d9, cha9 = dijkstra_tung_buoc(bt4, "A")

print("\n" + "=" * 66)
print("KẾT QUẢ: ĐƯỜNG ĐI NGẮN NHẤT TỪ A TỚI CÁC ĐỈNH CÒN LẠI")
print("=" * 66)
print(f"{'Đỉnh':>6} {'Khoảng cách':>13}   Đường đi")
print("-" * 66)
for v in bt4.dinh:
    duong = truy_vet(cha9, v)
    print(f"{v:>6} {d9[v]:>13}   {' -> '.join(duong)}")

=== BT9: ĐƯỜNG ĐI NGẮN NHẤT TỪ A (DIJKSTRA) ===
Đồ thị: 8 đỉnh, 14 cạnh

 Bước  Chọn u S                              D[A]    D[B]    D[C]    D[D]    D[E]    D[F]    D[G]    D[H]
--------------------------------------------------------------------------------------------------------
    0       - set()                             0     inf     inf     inf     inf     inf     inf     inf
    1       A {'A'}                             0       5       1     inf     inf     inf     inf     inf
    2       C {'A', 'C'}                        0       3       1       6      14     inf     inf      16
    3       B {'A', 'B', 'C'}                   0       3       1       5      14     inf     inf      16
    4       D {'D', 'A', 'B', 'C'}              0       3       1       5       8      15      13      16
    5       E {'D', 'C', 'B', 'E', 'A'}         0       3       1       5       8      15      12      16
    6       G {'D', 'C', 'B', 'E', 'A', 'G'}        0       3       1       5   

In [22]:
# ---- KIỂM THỬ BT9 ----
# 1) Khoảng cách tới A = 0
assert d9["A"] == 0
print("Test 1: d(A,A) = 0: PASS")

# 2) Mọi khoảng cách hữu hạn (đồ thị liên thông)
assert all(d9[v] < float("inf") for v in bt4.dinh)
print("Test 2: đồ thị liên thông -> mọi khoảng cách hữu hạn: PASS")

# 3) Đối chiếu với Bellman-Ford (thuật toán độc lập)
d_bf9 = bellman_ford(bt4, "A")
assert all(abs(d9[v] - d_bf9[v]) < 1e-9 for v in bt4.dinh)
print("Test 3: Dijkstra == Bellman-Ford cho MỌI đỉnh: PASS")

# 4) Đối chiếu với Floyd-Warshall
D9 = floyd_warshall(W4, V4)
iA = V4.index("A")
for j, v in enumerate(V4):
    assert D9[iA][j] == d9[v], f"Floyd {D9[iA][j]} != Dijkstra {d9[v]} tại {v}"
print("Test 4: Dijkstra == Floyd-Warshall cho MỌI đỉnh: PASS")

# 5) Đường đi truy vết phải HỢP LỆ và có tổng trọng số đúng bằng khoảng cách
for v in bt4.dinh:
    duong = truy_vet(cha9, v)
    assert duong[0] == "A" and duong[-1] == v
    for a, b in zip(duong, duong[1:]):
        assert b in [x for x, _ in bt4.ke[a]], f"{a}-{b} không phải cạnh"
    assert tong_trong_so(bt4, duong) == d9[v], f"{v}: tổng đường đi != khoảng cách"
print("Test 5: mọi đường đi truy vết hợp lệ và tổng trọng số khớp khoảng cách: PASS")

# 6) Kiểm chứng bất đẳng thức tam giác (tính đúng của khoảng cách ngắn nhất)
for u in bt4.dinh:
    for v, w in bt4.ke[u]:
        assert d9[v] <= d9[u] + w + 1e-9, f"vi phạm tam giác tại {u}->{v}"
print("Test 6: bất đẳng thức tam giác d[v] <= d[u] + w đúng với mọi cạnh: PASS")

# 7) Kiểm tra bằng vét cạn: khoảng cách = min tổng trọng số trên TẤT CẢ đường đi đơn
def vet_can_duong_di(g, s, t):
    tot_nhat = [float("inf")]
    def di(u, da_tham, tong):
        if u == t:
            tot_nhat[0] = min(tot_nhat[0], tong)
            return
        for v, w in g.ke[u]:
            if v not in da_tham:
                da_tham.add(v)
                di(v, da_tham, tong + w)
                da_tham.discard(v)
    di(s, {s}, 0)
    return tot_nhat[0]

for v in bt4.dinh:
    if v == "A":
        continue
    vc = vet_can_duong_di(bt4, "A", v)
    assert vc == d9[v], f"{v}: vét cạn {vc} != Dijkstra {d9[v]}"
print("Test 7: Dijkstra == vét cạn TẤT CẢ đường đi cho mọi đích: PASS")

# 8) Trọng số âm: Dijkstra không dùng được, Bellman-Ford thì được
g_am = DoThiCoHuong()
g_am.them_cung("A", "B", 1)
g_am.them_cung("B", "C", -2)
g_am.them_cung("A", "C", 5)
d_am = bellman_ford(g_am, "A")
assert d_am["C"] == -1, f"Bellman-Ford với trọng số âm phải cho -1, được {d_am['C']}"
print("Test 8: Bellman-Ford xử lý được trọng số âm (d(A,C) = -1): PASS")

Test 1: d(A,A) = 0: PASS
Test 2: đồ thị liên thông -> mọi khoảng cách hữu hạn: PASS
Test 3: Dijkstra == Bellman-Ford cho MỌI đỉnh: PASS
Test 4: Dijkstra == Floyd-Warshall cho MỌI đỉnh: PASS
Test 5: mọi đường đi truy vết hợp lệ và tổng trọng số khớp khoảng cách: PASS
Test 6: bất đẳng thức tam giác d[v] <= d[u] + w đúng với mọi cạnh: PASS
Test 7: Dijkstra == vét cạn TẤT CẢ đường đi cho mọi đích: PASS
Test 8: Bellman-Ford xử lý được trọng số âm (d(A,C) = -1): PASS


---
## Tổng kết

| Bài | Nội dung | Thuật toán | Kiểm thử chính |
|---|---|---|---|
| BT1 | Đồ thị 5 đỉnh, 7 cạnh; DFS từ 1 | DFS (stack + đệ quy), BFS | DFS stack == DFS đệ quy; BFS theo lớp; định lý bắt tay |
| BT2 | 4 hàm nhập/đọc/kiểm tra | — | ma trận đối xứng ↔ vô hướng; đọc file thật trên đĩa |
| BT3 | Đường đi 6 → 8 | BFS, DFS | đối chiếu bảng duyệt slide; BFS = đường đi ngắn nhất |
| BT4 | Ma trận, duyệt, đường đi A→H | DFS, BFS, Dijkstra | Dijkstra == Bellman-Ford == Floyd-Warshall == vét cạn |
| BT5 | Thành phần liên thông | BFS, DFS | BFS == DFS == union-find |
| BT6 | Cây bao trùm | BFS, DFS | n−1 cạnh, liên thông, không chu trình |
| BT7 | Cây khung nhỏ nhất | Kruskal | Kruskal == Prim == **vét cạn** (tối ưu) |
| BT8 | Ma trận trọng số + các bước Prim | Prim | Prim == Kruskal == vét cạn; mọi đỉnh bắt đầu |
| BT9 | Đường đi ngắn nhất từ A | Dijkstra | == Bellman-Ford == Floyd-Warshall == vét cạn |

### Nhận xét quan trọng

- **Đồ thị BT7, BT8, BT9 gần như cùng một hình** (8 đỉnh A–H): BT7 và BT9 có 14 cạnh (kể cả **C–H = 15**), BT8 có 13 cạnh (thiếu C–H). Vì vậy cây khung nhỏ nhất của BT7 và BT8 giống nhau (cùng 7 cạnh, tổng trọng số **14**), còn đường đi ngắn nhất A→H của BT4/BT9 dùng được cạnh C–H nên ngắn hơn.
- **Mọi kết quả đều được kiểm chứng bằng ít nhất 2 thuật toán độc lập**, và với cây khung/đường đi ngắn nhất thì kiểm chứng thêm bằng **vét cạn toàn bộ** — đây là bằng chứng chặt chẽ rằng kết quả là tối ưu, không chỉ là "một lời giải hợp lệ".
- **BT3 được xác nhận chéo** với bảng duyệt BFS/DFS trong slide, nên danh sách kề dùng ở đây là chắc chắn đúng.